# IID verification run — clean `repro` package (branch `rebuttal`)
**Upload-and-run on a GPU runtime.** Pavia single-class + multi-class IID
experiments (the paper's `iid_grid` figure): vs-n sweep and vs-rho sweep,
5 seeds, detectors AMF / GMM-Levin(multi) / L-DART / DART / L-LRao / LRao.

The fixed LRao is NATIVE here (config `lrao_input_norm: robust` = per-band
median/IQR front-end; L-LRao forced linear) — no monkeypatching. The runner
`repro.protocols.iid.run_iid` is the verbatim published pipeline with
imports cleaned; it writes metrics.json / scores.npz / loss curves /
per-run figures, all zipped at the end.

In [ ]:
!git clone -b rebuttal --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, sys, torch, yaml
sys.path.insert(0, '.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
assert os.path.exists('repro/data/pavia-u.mat')

In [ ]:
from repro.protocols.iid import run_iid
cfg_s = yaml.safe_load(open('repro/configs/iid_single.yaml'))
cfg_s.update(dataset='repro/data/pavia-u.mat', device=DEVICE,
             results_dir='results/iid_single', lrao_input_norm='robust')
print({k: cfg_s[k] for k in ('n_train_list', 'rho_list', 'seed')
       if k in cfg_s})

In [ ]:
run_iid(cfg_s, mode='single')

In [ ]:
cfg_m = yaml.safe_load(open('repro/configs/iid_multi.yaml'))
cfg_m.update(dataset='repro/data/pavia-u.mat', device=DEVICE,
             results_dir='results/iid_multi', lrao_input_norm='robust')
run_iid(cfg_m, mode='multi')

In [ ]:
import zipfile, os
with zipfile.ZipFile('iid_verification.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for d in ('results/iid_single', 'results/iid_multi'):
        for root, _, files in os.walk(d):
            for fn in files:
                z.write(os.path.join(root, fn))
print('zipped -> iid_verification.zip')
try:
    from google.colab import files
    files.download('iid_verification.zip')
except Exception as e:
    print('manual download:', e)